In [14]:
!pip install langchain

I0000 00:00:1765361507.334134 13876023 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


In [16]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage

In [2]:
load_dotenv()

True

In [4]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
model_with_search = model.bind_tools([{"google_search": {}}])

In [ ]:
response_with_search = model_with_search.invoke("What is the funding recieved in the last 3 years in Pondicherry University?")

In [7]:
import pprint

In [17]:
pprint.pprint(str(response.content))

('Pondicherry University receives funding from various sources, including '
 'grants from government agencies and sponsored projects. While specific '
 'consolidated figures for the total funding received in the last three '
 'financial years (2020-2021, 2021-2022, and 2022-2023) require a detailed '
 "review of the university's annual financial accounts, the relevant documents "
 'have been identified.\n'
 '\n'
 'Pondicherry University publishes its annual accounts, which would contain '
 'the detailed breakdown of receipts and income. The "ANNUAL ACCOUNTS '
 '2022-2023", "ANNUAL ACCOUNTS 2021-2022", and "ANNUAL ACCOUNTS 2020-2021" are '
 'available. These reports typically include sections like "Consolidated '
 'Receipt and Payments" or "Income & Expenditure Account - General" which '
 'would list grants, subsidies, and other funding sources.\n'
 '\n'
 'For instance, the 36th Annual Report for 2021-22 mentions that during that '
 'year, the university received 24 sponsored projects a

In [31]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.tools import tool

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    max_retries=2
)

@tool("retrieve_tool")
def retrieve_tool(query: str) -> str:
    """Search and return information from the internal vector database for a given query (str)."""
    search_result = vector_db.similarity_search(query=query)
    context = "\n\n".join([f"{result.page_content}\n" for result in search_result])

    return context

tools_list = [retrieve_tool,{"google_search": {}}]
model_with_search = model.bind(functions=tools_list)

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_db = QdrantVectorStore.from_existing_collection(
    url="http://localhost:6333",
    collection_name="PONDICHERRY_UNIVERSITY_INFO",
    embedding=embeddings
)

In [32]:
def generate(state: MessagesState):
    """Call the model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply respond to the user.
    """
    response = model_with_search.invoke(state['messages'])
    return {"message": response}

In [33]:
res = generate({"messages": "Hi how are you?"})
print(res)

Key 'title' is not supported in schema, ignoring


ChatGoogleGenerativeAIError: Invalid argument provided to Gemini: 400 Tool use with function calling is unsupported

In [ ]:
query = retrieve_tool.invoke({"query": "Which faculty has the highest experience?"})
print(query)

In [ ]:
def process_query(query: str):    
    search_result = vector_db.similarity_search(query=query)
    context = "\n\n".join([f"{result.page_content}\n" for result in search_result])

    SYSTEM_PROMPT = f"""
        You're an expert content specialist/consultant helping top management (HODs, Professors and even students) at Pondicherry University.
        YOU'RE FORCED TO DO A WEB SEARCH AND GATHER INFORMATION AS WELL GET THE INFORMATION FROM THE CONTEXT.
        Cross check the information, fill the data gaps and correct the information if needed, finally merge the information.
        Output the information in great details and use tabular structure whenever and whereever needed, but the output should be very CLEAN and READABLE.
        DON'T MAKE ANY MISTAKE
        INTERNAL DATABASE CONTEXT: {context}
    """

    messages = [
        ("system",SYSTEM_PROMPT),
        ("human", query),
    ]

    llm_response = model_with_search.invoke(messages)
    return llm_response.content